[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Carlo-Broschi/statistics-python-for-students/blob/main/04_%E7%B5%B1%E8%A8%88_2%E7%B4%9A/06_%E9%87%8D%E5%9B%9E%E5%B8%B0%E5%88%86%E6%9E%90.ipynb)

> Colabで実行可。最初に「① セットアップ」セルを実行（Colabはデータを自動生成、ローカルは何もしない）。

In [ ]:
# === ① セットアップ（最初に実行してください）===============================
# Colabなどデータが無い環境では、教材と同一の合成データを自動生成します。
# ローカル/Codespacesで data/ がある場合は何もしません（再現性のため乱数seedは固定）。
import os
if not os.path.exists('../data/students_scores.csv'):   # データが無い環境（Colab等）か判定
    os.makedirs('/content/nb', exist_ok=True); os.chdir('/content/nb')  # ../data を使えるよう作業場所を作る
    os.makedirs('../data', exist_ok=True)               # データ保存先フォルダを作る
    import numpy as np, pandas as pd                     # 数値計算と表データのライブラリ
    D = '../data'; rng = np.random.default_rng(42)       # 保存先と、結果を固定する乱数生成器
    # --- 生徒120人の成績データを作る ---
    n = 120; cls = rng.choice(['A','B','C'], n)          # 人数とクラス
    math = np.clip(rng.normal(65,15,n),0,100).round().astype(int)        # 数学（0〜100点）
    eng = np.clip(0.6*math+rng.normal(28,10,n),0,100).round().astype(int) # 英語（数学と相関）
    jp = np.clip(rng.normal(70,12,n),0,100).round().astype(int)          # 国語
    hrs = np.clip(rng.normal(1.5,0.8,n)+math/100,0,None).round(1)        # 勉強時間
    pd.DataFrame({'生徒ID':[f'S{i:03d}' for i in range(1,n+1)],'クラス':cls,
        '数学':math,'英語':eng,'国語':jp,'勉強時間':hrs}).to_csv(f'{D}/students_scores.csv',index=False,encoding='utf-8-sig')
    # --- 30日間の天気データ ---
    days = pd.date_range('2026-06-01', periods=30)       # 6月の30日分の日付
    temp = (24+4*np.sin(np.arange(30)/5)+rng.normal(0,2,30)).round(1)    # 気温
    rain = np.clip(rng.gamma(1.2,4,30)*(rng.random(30)<0.4),0,None).round(1)  # 降水量
    pd.DataFrame({'日付':days.strftime('%Y-%m-%d'),'気温':temp,'降水量':rain}).to_csv(f'{D}/weather.csv',index=False,encoding='utf-8-sig')
    # --- BtoB商談400件 ---
    m = 400; inds = rng.choice(['製造','IT','小売','医療','金融','建設'],m,p=[.25,.2,.18,.12,.15,.1])  # 業界
    chs = ['展示会','Web広告','メルマガ','紹介','テレアポ']; ch = rng.choice(chs,m,p=[.2,.3,.2,.15,.15])  # 獲得チャネル
    deal = (rng.lognormal(13,0.7,m)).round(-3).astype(int)   # 商談金額
    wr = {'展示会':.35,'Web広告':.18,'メルマガ':.12,'紹介':.45,'テレアポ':.15}  # チャネル別の受注率
    won = np.array([rng.random()<wr[c] for c in ch])         # 受注したか（チャネルで確率を変える）
    dt = pd.to_datetime('2026-01-01')+pd.to_timedelta(rng.integers(0,180,m),unit='D')  # 受注日
    pd.DataFrame({'商談ID':[f'D{i:04d}' for i in range(1,m+1)],'受注日':dt.strftime('%Y-%m-%d'),
        '業界':inds,'獲得チャネル':ch,'商談金額':deal,'担当者':rng.choice(['田中','鈴木','佐藤','高橋'],m),
        '受注':won.astype(int)}).to_csv(f'{D}/sales_btob.csv',index=False,encoding='utf-8-sig')
    # --- 月×チャネルのマーケ実績 ---
    months = pd.date_range('2026-01-01',periods=6,freq='MS').strftime('%Y-%m')  # 6か月分
    base = {'展示会':2000,'Web広告':12000,'メルマガ':6000,'紹介':800,'テレアポ':1500}  # 表示回数の目安
    ctr = {'展示会':.08,'Web広告':.03,'メルマガ':.05,'紹介':.2,'テレアポ':.12}   # クリック率
    cvr = {'展示会':.12,'Web広告':.04,'メルマガ':.06,'紹介':.25,'テレアポ':.09}   # 獲得率
    cpc = {'展示会':30,'Web広告':80,'メルマガ':5,'紹介':0,'テレアポ':200}        # クリック単価
    rows = []
    for mo in months:                                    # 月ごと
        for c in chs:                                    # チャネルごとに実績を作る
            imp = int(base[c]*rng.uniform(0.8,1.3)); clk = int(imp*ctr[c]*rng.uniform(0.8,1.2))
            cv = int(clk*cvr[c]*rng.uniform(0.7,1.3)); cost = int(clk*cpc[c]*rng.uniform(0.9,1.1))
            rows.append([mo,c,imp,clk,cv,cost])
    pd.DataFrame(rows,columns=['月','チャネル','表示回数','クリック数','獲得数','費用']).to_csv(f'{D}/web_marketing.csv',index=False,encoding='utf-8-sig')
    # --- A/Bテストの申込ログ ---
    ab = []
    for grp,p,size in [('A_青ボタン',0.082,2400),('B_緑ボタン',0.104,2380)]:  # 2群の申込率と人数
        for c in (rng.random(size)<p): ab.append([grp,int(c)])   # 各訪問が申込んだか
    pd.DataFrame(ab,columns=['グループ','申込']).sample(frac=1,random_state=1).reset_index(drop=True).to_csv(f'{D}/ab_test.csv',index=False,encoding='utf-8-sig')
    print('✅ 合成データを生成しました（Colab環境）')
else:
    print('✅ data/ を検出しました（ローカル/Codespaces）')

# 統計2級-06. 重回帰分析

**できるようになること**
- 複数の要因で結果を説明・予測できる
- 偏回帰係数・決定係数・係数のp値を読める
- ダミー変数・多重共線性(VIF)を扱える

**前提**：統計3級02（相関と回帰）　/　**所要**：約35分　/　**レベル**：統計検定2級相当

In [ ]:
import numpy as np               # 数値計算
import pandas as pd              # 表データ
import matplotlib.pyplot as plt  # グラフ描画
import statsmodels.formula.api as smf   # 回帰分析
plt.rcParams['axes.unicode_minus'] = False       # マイナス記号の文字化けを防ぐ
df = pd.read_csv('../data/students_scores.csv')   # 生徒データを読み込む
df.head()                        # 先頭5行を確認

### 使うデータ：`students_scores.csv`（架空の生徒120人の成績）
1行＝生徒1人。数学・英語・国語は0〜100点、勉強時間は1日あたりの時間です。

| 生徒ID | クラス | 数学 | 英語 | 国語 | 勉強時間 |
|---|---|---|---|---|---|
| S001 | A | 52 | 58 | 49 | 2.0 |
| S002 | C | 80 | 79 | 74 | 3.4 |
| S003 | B | 40 | 65 | 91 | 2.0 |

（下のセルの `df.head()` で先頭5行を実際に確認できます）

---

### 使うデータ：`sales_btob.csv`（架空のBtoB商談400件）
1行＝商談1件。`受注`は 1=成約 / 0=失注。

| 商談ID | 受注日 | 業界 | 獲得チャネル | 商談金額 | 担当者 | 受注 |
|---|---|---|---|---|---|---|
| D0001 | 2026-01-15 | 小売 | 紹介 | 1,211,000 | 佐藤 | 0 |
| D0002 | 2026-02-05 | 医療 | テレアポ | 400,000 | 田中 | 0 |
| D0003 | 2026-02-19 | IT | 紹介 | 542,000 | 高橋 | 1 |

## 1. 単回帰の復習 → 重回帰へ

`数学` を `英語`・`国語`・`勉強時間` から説明します。
$$ 数学 = b_0 + b_1\,英語 + b_2\,国語 + b_3\,勉強時間 $$

`statsmodels` を使うと、表計算ソフトのような回帰の出力が得られます。

> **数IIIメモ**：重回帰の係数は内部で**行列の計算（大学範囲）**で求まりますが、**手計算は不要**。ここでは『各要因が結果をどれだけ動かすか（偏回帰係数）』の**読み方**に集中します。

In [ ]:
model = smf.ols('数学 ~ 英語 + 国語 + 勉強時間', data=df).fit()  # 重回帰モデルを当てはめる
print(model.summary())                     # 回帰の詳しい結果を表示

## 2. 出力の読み方（2級の頻出ポイント）

- **coef（偏回帰係数）**：他の変数を一定にしたとき、その変数が1増えると数学が何点変わるか
- **R-squared（決定係数）**：当てはまりの良さ（0〜1、1に近いほど良い）
- **Adj. R-squared（自由度調整済み）**：変数の数を考慮した決定係数。変数比較に使う
- **P>|t|（回帰係数の検定）**：その変数が本当に効いているか（0.05未満で有意）

In [ ]:
print('決定係数 R^2      :', round(model.rsquared, 3))       # 当てはまりの良さ
print('自由度調整済み R^2:', round(model.rsquared_adj, 3))   # 変数の数を考慮した決定係数
print('\n偏回帰係数:')
print(model.params.round(3))               # 各変数の係数
print('\n各係数のp値:')
print(model.pvalues.round(4))              # 各係数が有意か（0.05未満で有意）

## 3. 予測してみる

In [ ]:
new = pd.DataFrame({'英語':[70], '国語':[75], '勉強時間':[2.0]})  # 予測したい人の値
print('予測される数学の点数:', round(model.predict(new)[0], 1))   # モデルで予測

## 4. ダミー変数（質的データを回帰に入れる）

`クラス`(A/B/C)のような質的変数は、0/1の**ダミー変数**にして投入します。
`C(クラス)` と書くと自動でダミー化されます。

In [ ]:
model2 = smf.ols('数学 ~ 英語 + C(クラス)', data=df).fit()  # 質的変数クラスをダミー化して投入
print(model2.params.round(2))
print('→ [T.B],[T.C] は基準クラスAと比べた差を表す')

## 5. 多重共線性（マルチコ）

説明変数どうしが強く相関していると、係数が不安定になる問題。
**VIF**（分散拡大係数）で確認します。VIFが10を超えると要注意。

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor  # VIF計算
X = df[['英語', '国語', '勉強時間']].copy()  # 説明変数
X['const'] = 1                             # 切片用の列
# 各変数のVIF（分散拡大係数）を計算
vif = pd.Series(
    [variance_inflation_factor(X.values, i) for i in range(X.shape[1])],
    index=X.columns)
print(vif.round(2))
print('→ const以外が10未満なら多重共線性は問題なし')

> **よくある間違い**：決定係数R²が高い＝良いモデルとは限らない（過学習・多重共線性に注意）。また 0/1 の結果（受注/生存など）には重回帰でなく**ロジスティック回帰**が適切。

## 6. 残差プロットと自由度調整済み決定係数

回帰が妥当かは**残差**（実測−予測）を見るとわかります。残差に偏りや曲線が見えなければ良いモデルです。

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.api as sm
df = pd.read_csv('../data/students_scores.csv')
X = sm.add_constant(df[['勉強時間']]); y = df['数学']
res = sm.OLS(y, X).fit()
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(res.fittedvalues, res.resid, alpha=.5); ax.axhline(0, color='k', ls='--')
ax.set_xlabel('予測値'); ax.set_ylabel('残差'); ax.set_title('残差プロット（構造が無ければOK）')
plt.show()
print(f'決定係数 R² = {res.rsquared:.3f} / 自由度調整済み R² = {res.rsquared_adj:.3f}')

> 説明変数を増やすと**素の R² は必ず上がります**（意味のない変数でも）。だからモデルの良し悪しを比べるときは、変数の数で割り引いた**自由度調整済み R²** を使います。
>
> 残差プロットに曲線・ラッパ状の広がり（不等分散）が見えたら、モデルの作り直しを検討します。

## 確認テスト（自動採点）

`ans = None` を**自分の計算で置き換えて実行**すると、その場で正誤が出ます（答えは表示されません）。

In [ ]:
# 採点用の関数を実行（答え合わせに使うだけ）
def _check(name, got, want, tol=1e-2):
    if got is None:
        print(f'⬜ {name}: まだ入力されていません（ans を埋めてね）'); return
    ok = abs(got - want) <= tol
    print(('✅ 正解！ ' if ok else '❌ もう一度 ') + f'{name}: あなたの答え = {got}')

In [ ]:
import statsmodels.formula.api as smf
r = df['数学'].corr(df['英語'])
# Q1: 単回帰 数学~英語 の決定係数は相関係数の2乗。ans に r**2 を入れよう
ans = None   # 例: r**2
_check('Q1 R^2 = r^2', ans, smf.ols('数学 ~ 英語', data=df).fit().rsquared)

In [ ]:
# Q2: 3カテゴリの質的変数をダミー変数にすると何本？（基準を1つ除く）を ans に
ans = None   # 3 - 1
_check('Q2 ダミーの本数', ans, 3-1)

In [ ]:
# Q3: 説明変数を1つ増やすと「素の決定係数R²」は必ず増える(=1)か、減ることもある(=0)か。ans に
ans = None   # ヒント: 素のR²は説明変数を足すと必ず増える
_check('Q3 R²の性質', ans, 1)

---
## 練習問題

**問1.** `英語` を `数学`・`国語`・`勉強時間` で説明する重回帰を作り、
決定係数と、有意な（p<0.05）説明変数を答えよう。

**問2.** `sales_btob.csv` で `商談金額` を `業界`（ダミー）で説明する回帰を作り、
どの業界が金額を押し上げ／押し下げているか読み取ろう。

**問3.** 問1のモデルに`クラス`のダミーを加えると自由度調整済みR²は上がる？下がる？確かめよう。

**問4.** `数学` を `勉強時間` だけで回帰した場合と、`勉強時間`＋`国語` で回帰した場合で、素のR²と自由度調整済みR²を比べよう。どちらが「変数を足した効果」を正しく評価している？

In [ ]:
# 問1


In [ ]:
# 問2
btob = pd.read_csv('../data/sales_btob.csv')   # 商談データを読み込む

> **解答例は別ページにまとめています**（ネタバレ防止）。
> 自分で解いてから **[06_重回帰分析 の解答例を開く](https://github.com/Carlo-Broschi/statistics-python-for-students/blob/main/%E8%A7%A3%E7%AD%94%E9%9B%86/04_%E7%B5%B1%E8%A8%88_2%E7%B4%9A/06_%E9%87%8D%E5%9B%9E%E5%B8%B0%E5%88%86%E6%9E%90.md)**

## 用語集 ＆ チートシート

| 用語 | 意味 |
|---|---|
| 偏回帰係数 | 他を一定にしたときの効き目 |
| 決定係数 R² | 当てはまりの良さ |
| 自由度調整済みR² | 変数の数を考慮 |
| ダミー変数 | 質的変数を0/1に |
| 多重共線性(VIF) | 説明変数どうしの強い相関 |

In [ ]:
# チートシート（重回帰）
import statsmodels.formula.api as smf
m = smf.ols('数学 ~ 英語 + 国語', data=df).fit()
print(m.rsquared, m.rsquared_adj)   # 決定係数・自由度調整済み
print(m.params)                     # 偏回帰係数
print(m.pvalues)                    # 係数のp値

## 完了ログ

このノートを終えたら下のセルを実行すると、学習の完了が記録されます。
**学習者コードは最初に一度だけ設定**すれば、以降は全ノートで自動送信されます（名前の再入力は不要）。

- Colab：左の鍵アイコン（シークレット）で `STUDENT_CODE` に配布コードを登録（1回だけ）
- ローカル：環境変数 `STUDENT_CODE` を設定（または初回に画面入力すると保存され、次回から不要）

In [ ]:
# === 完了ログ ===  学習者コードは最初に一度だけ設定すれば、全ノートで自動送信されます。
import os, datetime, pathlib
try:
    import requests
except ImportError:
    requests = None

def _student_code():
    try:                                          # Colab: 鍵アイコンで STUDENT_CODE を登録
        from google.colab import userdata
        c = userdata.get('STUDENT_CODE')
        if c: return c.strip()
    except Exception:
        pass
    c = os.environ.get('STUDENT_CODE')            # ローカル: 環境変数
    if c: return c.strip()
    p = pathlib.Path.home() / '.student_code'      # 保存ファイル
    if p.exists(): return p.read_text().strip()
    try:                                           # 最後の手段: 入力して保存（次回から不要）
        c = input('学習者コードを入力（配布されたもの）: ').strip()
        if c: p.write_text(c); return c
    except Exception:
        pass
    return ''

CODE = _student_code()
LOG_URL = ""      # 配布時に設定
LOG_TOKEN = ""    # 配布時に設定
NOTEBOOK = "04_統計_2級/06_重回帰分析"

if CODE and LOG_URL and requests:
    try:
        requests.post(LOG_URL, json={"token": LOG_TOKEN, "code": CODE, "notebook": NOTEBOOK,
                      "time": datetime.datetime.now().isoformat(timespec="seconds")}, timeout=10)
        print(f"記録しました: {CODE} / {NOTEBOOK}")
    except Exception as e:
        print("記録に失敗しました（URL/ネットワークを確認）:", e)
elif not CODE:
    print("学習者コード未設定。Colabは鍵アイコンで STUDENT_CODE を登録、ローカルは環境変数 STUDENT_CODE を設定してください。")
else:
    print(f"{NOTEBOOK}: LOG_URL/LOG_TOKEN が未設定です（配布時に設定されます）。")